# Clasp weighted-sequence: random-configuration value distribution

This notebook analyzes 1,000 unique random configurations on every training instance. Quantile seed 0 is the deterministic median of the ACLib EPM; the other five seeds are reproducible stochastic draws. Keeping these axes separate prevents target stochasticity from being confused with variation across configurations or instances.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from IPython.display import display

CANDIDATES = [
    Path.cwd().resolve(),
    (Path.cwd() / 'experiments/aclib/clasp_weighted-sequence_surrogate/06_sample').resolve(),
]
HERE = next((path for path in CANDIDATES if (path / 'results').exists()), CANDIDATES[0])
ACLIB_ROOT = next(
    path for path in [HERE.parents[1], (Path.cwd() / 'experiments/aclib').resolve()]
    if (path / 'random_sampling_analytics.py').exists()
)
if str(ACLIB_ROOT) not in sys.path:
    sys.path.insert(0, str(ACLIB_ROOT))

from random_sampling_analytics import *
results = load_sampling_results(HERE)
display(overview_table(results))

## 1. Distribution of configuration values

Histogram and empirical CDF: each observation is one configuration's mean PAR10 over all 240 training instances on the deterministic median surface. Logarithmic performance axes make both fast configurations and PAR10-heavy configurations visible.

In [ ]:
display(configuration_distribution_table(results))
plot_value_distribution(results)
plt.show()

Best/worst tables: the configurations are ranked by their full-training mean PAR10. The accompanying instance spread and timeout fraction show whether a good mean is broad or driven by a subset of instances.

In [ ]:
best, worst = ranked_configurations(results, n=10)
print('Ten best sampled configurations')
display(best)
print('Ten worst sampled configurations')
display(worst)

## 2. Variability across training instances

Scatter plot and histogram: every point is one configuration. The scatter compares its mean PAR10 with its standard deviation across instances; color is its timeout fraction. The second panel shows relative instance variability, so configurations at different performance scales can be compared.

In [ ]:
display(within_configuration_variability_table(results))
plot_instance_variability(results)
plt.show()

## 3. Variability of the stochastic target surrogate

Scatter plot and histogram: the left panel compares quantile seed 0 with the mean of five stochastic EPM draws for the same configuration and all training instances. The right panel measures draw-to-draw variability. A weak rank correlation would mean that fixing the target to its median materially changes the benchmark landscape.

In [ ]:
display(quantile_seed_table(results))
display(stochastic_variability_table(results))
plot_stochastic_variability(results)
plt.show()

## 4. Instance effects

Histogram and bar chart: instance difficulty is the mean PAR10 over all sampled configurations at quantile seed 0. The right panel identifies the 20 instances that dominate the high-cost tail.

In [ ]:
display(instance_table(results, n=15))
plot_instance_effects(results, n_hardest=20)
plt.show()

## 5. Sources of variation and sample stability

Variance table: these rows deliberately retain their stated aggregation level—between configurations, between instances within a configuration, and between stochastic EPM draws of full-training means. They diagnose scale, but should not be read as percentages of one classical ANOVA decomposition.

Convergence plot: the running best, median, and q10–q90 range show whether 1,000 random configurations are enough for stable distribution summaries.

In [ ]:
display(variance_decomposition(results))
plot_sampling_convergence(results)
plt.show()

## Interpretation guide

- A wide configuration distribution means the target distinguishes solver configurations strongly.
- Large within-configuration instance variation means adaptive intensification can produce optimistic early incumbents when coverage is small.
- Large stochastic draw variation or a weak median-versus-stochastic rank correlation means deterministic quantile seed 0 defines a meaningfully different optimization landscape from the official stochastic ACLib scenario.
- A stable median/q10/q90 with a still-improving running best means the distribution estimate is reliable even though more random samples could still find a better extreme.